# Tool Calling
We’ll give both models a fake get_order_status tool and let them decide to call it. You write the tool, run it yourself, and send the result back. Each provider needs two requests (one for the call, one for the final answer).

Question initially tested was successful for both providers. The current status reflects the testing stage of number of tool calls by each model when the initial question is not answered.

In [4]:
import os, json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from openai import OpenAI

load_dotenv(override=True)
GEMINI_MODEL = os.environ["GEMINI_MODEL"]
GROQ_MODEL = os.environ["GROQ_MODEL"]
gemini_client = genai.Client()
groq_client = OpenAI(api_key=os.environ["GROQ_API_KEY"], base_url=os.environ["GROQ_BASE_URL"])

ORDERS = {
    "A-10492": {"status": "refund_pending", "charges": [49.99, 49.99], "currency": "USD"},
    "B-20871": {"status": "shipped", "charges": [15.00], "currency": "USD"},
}

def get_order_status(order_id: str) -> dict:
    return ORDERS.get(order_id, {"error": f"order {order_id} not found"})

TOOL_NAME = "get_order_status"
TOOL_DESC = "Look up the status and charges for an order by its ID."
TOOL_PARAMS = {
    "type": "object",
    "properties": {"order_id": {"type": "string", "description": "e.g. A-10492"}},
    "required": ["order_id"],
}
QUESTION = "Compare the statuses of A-10492 and B-20871."

One schema dict, reused by both providers. Each wraps it differently, as the next cells show.

In [5]:
tool = types.Tool(function_declarations=[
    {"name": TOOL_NAME, "description": TOOL_DESC, "parameters": TOOL_PARAMS}
])
config = types.GenerateContentConfig(
    tools=[tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

contents = [types.Content(role="user", parts=[types.Part(text=QUESTION)])]
r1 = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)

call = r1.function_calls[0]
print("Gemini wants:", call.name, dict(call.args))

result = get_order_status(**call.args)
contents.append(r1.candidates[0].content)   # append the model's WHOLE turn as-is
contents.append(types.Content(role="user", parts=[
    types.Part.from_function_response(name=call.name, response={"result": result})
]))
r2 = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
print(r2.text)

Gemini wants: get_order_status {'order_id': 'A-10492'}


None


Without automatic_function_calling disabled, the SDK would run your function for you. We turn that off so you can see, and later control, each step. Appending the model’s turn as-is, instead of rebuilding it, matters: newer Gemini models attach a “thought signature” to their turn that they expect back (see Google’s Thought signatures docs).

In [6]:
tools = [{"type": "function",
          "function": {"name": TOOL_NAME, "description": TOOL_DESC, "parameters": TOOL_PARAMS}}]
messages = [{"role": "user", "content": QUESTION}]

r1 = groq_client.chat.completions.create(
    model=GROQ_MODEL, messages=messages, tools=tools, tool_choice="auto",
    extra_body={"reasoning_effort": "low"},
)
msg = r1.choices[0].message
tc = msg.tool_calls[0]
args = json.loads(tc.function.arguments)        # arguments arrive as a JSON STRING
print("Groq wants:", tc.function.name, args)

messages.append(msg)
messages.append({"role": "tool", "tool_call_id": tc.id,
                 "content": json.dumps(get_order_status(**args))})
r2 = groq_client.chat.completions.create(
    model=GROQ_MODEL, messages=messages, tools=tools,
    extra_body={"reasoning_effort": "low"},
)
print(r2.choices[0].message.content)

Groq wants: get_order_status {'order_id': 'A-10492'}
None


Were the tool names and arguments identical across providers?


Did both final answers correctly say the order was charged twice (two charges of 49.99) and give the status?


How does each represent “the model’s turn” and “the tool’s result” in the message history? That mapping is the core of your provider interface.

# Test the tool calling for each model in a loop until success.

In [7]:
def run_gemini_tool_loop(question: str, max_rounds: int = 5) -> str:
    """Run Gemini until it answers in plain text, executing any tools it requests."""
    # Conversation history: starts with just the user's question
    contents = [types.Content(role="user", parts=[types.Part(text=question)])]

    # Cap the rounds so a confused model can't loop forever and burn free-tier quota
    for round_num in range(1, max_rounds + 1):
        resp = gemini_client.models.generate_content(
            model=GEMINI_MODEL, contents=contents, config=config
        )

        # function_calls is None when the model answered in text, so default to []
        calls = resp.function_calls or []
        print(f"Gemini round {round_num}: {len(calls)} tool call(s)")

        # No tool calls means this is the final answer, so stop looping
        if not calls:
            return resp.text

        # Add the model's whole turn to the history unchanged
        # (newer models attach metadata that must be passed back as-is)
        contents.append(resp.candidates[0].content)

        # Run EVERY requested tool, not just the first, and collect the results
        result_parts = []
        for call in calls:
            result = get_order_status(**call.args)
            result_parts.append(
                types.Part.from_function_response(
                    name=call.name, response={"result": result}
                )
            )

        # Send all results back together as a single user turn
        contents.append(types.Content(role="user", parts=result_parts))

    raise RuntimeError(f"No final answer after {max_rounds} rounds")

QUESTION = "Compare the statuses of A-10492 and B-20871."
print(run_gemini_tool_loop(QUESTION))

Gemini round 1: 2 tool call(s)
Gemini round 2: 0 tool call(s)
Here is the comparison of orders A-10492 and B-20871:

* **Order A-10492:**
  * **Status:** Refund pending
  * **Charges:** $49.99 (two charges of $49.99 each)

* **Order B-20871:**
  * **Status:** Shipped
  * **Charges:** $15.00 (single charge of $15.00)


That matches the parallel behavior I expected from Gemini: it asked for both orders in a single round (2 calls), then answered in plain text on round 2, so the whole comparison took two requests.

The facts are all correct: refund_pending with two charges of 49.99 for A-10492, and shipped with one charge of 15.00 for B-20871. One small wording issue is that the first order’s “Charges: $49.99” line reads like a total when the actual total is $99.98. The parenthetical rescues it, but it shows why an eval suite later needs to check specific facts and not just whether the answer looks plausible.

In [8]:
def run_groq_tool_loop(question: str, max_rounds: int = 5) -> str:
    """Run Groq until it answers in plain text, executing any tools it requests."""
    # Conversation history in OpenAI chat format
    messages = [{"role": "user", "content": question}]

    # Same safety cap as the Gemini loop, to protect the daily quota
    for round_num in range(1, max_rounds + 1):
        resp = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto",                          # let the model decide whether to call a tool
            extra_body={"reasoning_effort": "low"},      # keep reasoning tokens (and quota use) down
        )
        msg = resp.choices[0].message

        # tool_calls is None when the model answered in text, so default to []
        calls = msg.tool_calls or []
        print(f"Groq round {round_num}: {len(calls)} tool call(s)")

        # No tool calls means this is the final answer, so stop looping
        if not calls:
            return msg.content

        # Add the model's turn (including its tool_calls) to the history
        messages.append(msg)

        # Run every requested tool and reply once per call
        for tc in calls:
            args = json.loads(tc.function.arguments)     # arguments arrive as a JSON string
            result = get_order_status(**args)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,                   # ties this result to the call that requested it
                "content": json.dumps(result),           # tool results must be strings
            })

    raise RuntimeError(f"No final answer after {max_rounds} rounds")

print(run_groq_tool_loop(QUESTION))

Groq round 1: 1 tool call(s)
Groq round 2: 1 tool call(s)
Groq round 3: 0 tool call(s)
**Order Status Comparison**

| Order ID | Current Status | Charges (USD) | Notes |
|----------|----------------|---------------|-------|
| **A-10492** | **refund_pending** | $49.99 × 2 (total $99.98) | The refund is awaiting processing. No shipment has occurred. |
| **B-20871** | **shipped** | $15.00 (total $15.00) | The item has already been dispatched to the customer. |

**Key Differences**

1. **Status**  
   - *A‑10492* is still in a refund‑pending state, meaning the purchase is being reversed.  
   - *B‑20871* has progressed past payment and is now shipped.

2. **Financial Impact**  
   - *A‑10492* involves a larger total charge ($99.98) that will be returned to the buyer once the refund is completed.  
   - *B‑20871* reflects a single charge of $15.00 that the customer has already paid and will retain after receiving the product.

3. **Next Steps**  
   - For *A‑10492*, monitor the refund proce

The Groq result matches my prediction. It used three rounds with one call each, while Gemini needed two rounds with two parallel calls. That makes Groq 50% more requests for the same question, and each later round resends the whole conversation, so its token cost grows too. On a free tier with an 8,000 tokens-per-minute cap, that adds up.

There’s a quality difference worth noting as well:

Groq got the arithmetic right. It stated the total ($99.98) outright, where Gemini’s “Charges: $49.99” line was ambiguous.
Groq also invented details. “No shipment has occurred,” the “Next Steps” advice, and “confirm receipt with the customer” appear nowhere in the tool data. The model filled gaps with plausible-sounding claims.

That second point is a good eval case for the README. A grounding check should flag claims that go beyond the tool output, and a system prompt like “answer only from the tool results” is the first thing to try.

Notebook stage done. You’ve now seen the raw differences for plain generation, structured output, and tool calling, and noted them for the abstraction layer to hide. The scaffold is next.